# TRIDENT Multi-Model ML Tournament
## 8-State Disaster Trauma Classification Engine

**Objective**: Competitive tournament across 4 ML architectures with 3-stage pruning protocol.

**Stage Pipeline**:
- **Stage 1 (10% data)**: Screen all 4 models (XGBoost, LightGBM, 1D-CNN, MLP) → Top 2 advance
- **Stage 2 (35% data)**: Semifinal with top 2 → Winner selected
- **Stage 3 (100% data)**: Champion trained on full dataset → Final artifact export

**GPU**: Enabled (auto-detects Kaggle GPU tier)

**Runtime**: ~20-40 minutes (GPU tier)

---

## Cell 1: Setup & Configuration

In [ ]:
import os
import sys
import time
import argparse
import datetime
import logging
import gc
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Dict, Any, List

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, log_loss
from sklearn.model_selection import GroupShuffleSplit
import joblib

# ML libraries
import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

print(f"✅ All imports successful")

# ─── GPU Detection ───────────────────────────────────────────────────────────
GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device('cuda' if GPU_AVAILABLE else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "N/A"

print(f"\n🖥️  Compute Environment:")
print(f"   GPU Available: {GPU_AVAILABLE}")
print(f"   Device: {DEVICE}")
print(f"   GPU Model: {GPU_NAME}")
if GPU_AVAILABLE:
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ─── Kaggle Paths ───────────────────────────────────────────────────────────
DATA_PATH = '/kaggle/input/trident-wearable-telemetry/fused_training_matrix.parquet'
OUTPUT_DIR = '/kaggle/working/'
DOCS_DIR = '/kaggle/working/'

# ─── Logging Configuration ───────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# ─── Constants ───────────────────────────────────────────────────────────────
FEATURE_COLS = [
    'bpm_current', 'bpm_mean_15s', 'bpm_std_15s',
    'pulse_amp_decay', 'spo2_current', 'spo2_slope',
    'acc_mag_max_3s', 'motion_stasis', 'gsr_stress_norm',
    'gsr_slope', 'fall_flag', 'gps_displacement_15s',
    'ksc', 'ssi', 'eim',
]

STATE_NAMES = {
    0: 'STABLE', 1: 'ELEVATED_STRESS', 2: 'HYPOTHERMIA_RISK',
    3: 'ASPIRATION_HYPOXIA', 4: 'HEMORRHAGIC_SHOCK',
    5: 'CONCUSSIVE_STASIS', 6: 'CRUSH_ENTRAPMENT', 7: 'CARDIAC_ARREST'
}

NUM_CLASSES = len(STATE_NAMES)

print(f"\n📊 Dataset Configuration:")
print(f"   Features: {len(FEATURE_COLS)}")
print(f"   Classes: {NUM_CLASSES}")
print(f"   Data Path: {DATA_PATH}")

## Cell 2: Model Architectures

### Base Model Interface
Define abstract base class for all 4 model types.

In [ ]:
# ─── Base Model Interface ────────────────────────────────────────────────────
class BaseModel(ABC):
    name: str

    @abstractmethod
    def train(self, X_train: pd.DataFrame, y_train: pd.Series, X_val: pd.DataFrame, y_val: pd.Series) -> dict:
        pass

    @abstractmethod
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        pass

    @abstractmethod
    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        pass

    @abstractmethod
    def save(self, path: str):
        pass

    @abstractmethod
    def get_model_size_mb(self) -> float:
        pass


# ─── XGBoost Model ───────────────────────────────────────────────────────────
class XGBoostModel(BaseModel):
    def __init__(self):
        self.name = 'XGBoost'
        self.model = None

    def train(self, X_train, y_train, X_val, y_val) -> dict:
        tree_method = 'gpu_hist' if GPU_AVAILABLE else 'hist'
        self.model = xgb.XGBClassifier(
            tree_method=tree_method,
            objective='multi:softprob',
            num_class=NUM_CLASSES,
            eval_metric='mlogloss',
            early_stopping_rounds=15,
            n_estimators=500,
            use_label_encoder=False,
            random_state=42
        )
        
        start_time = time.time()
        self.model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        train_time = time.time() - start_time
        return {"train_time": train_time, "best_iteration": self.model.best_iteration}

    def predict(self, X) -> np.ndarray:
        return self.model.predict(X)

    def predict_proba(self, X) -> np.ndarray:
        return self.model.predict_proba(X)

    def save(self, path: str):
        joblib.dump(self.model, path)

    def get_model_size_mb(self) -> float:
        temp_path = '/tmp/temp_xgb.pkl'
        joblib.dump(self.model, temp_path)
        size = os.path.getsize(temp_path) / (1024 * 1024)
        if os.path.exists(temp_path):
            os.remove(temp_path)
        return size


# ─── LightGBM Model ──────────────────────────────────────────────────────────
class LightGBMModel(BaseModel):
    def __init__(self):
        self.name = 'LightGBM'
        self.model = None

    def train(self, X_train, y_train, X_val, y_val) -> dict:
        device_type = 'gpu' if GPU_AVAILABLE else 'cpu'
        self.model = lgb.LGBMClassifier(
            objective='multiclass',
            num_class=NUM_CLASSES,
            n_estimators=500,
            device=device_type,
            random_state=42
        )
        
        start_time = time.time()
        self.model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
        )
        train_time = time.time() - start_time
        return {"train_time": train_time, "best_iteration": self.model.best_iteration_}

    def predict(self, X) -> np.ndarray:
        return self.model.predict(X)

    def predict_proba(self, X) -> np.ndarray:
        return self.model.predict_proba(X)

    def save(self, path: str):
        joblib.dump(self.model, path)

    def get_model_size_mb(self) -> float:
        temp_path = '/tmp/temp_lgb.pkl'
        joblib.dump(self.model, temp_path)
        size = os.path.getsize(temp_path) / (1024 * 1024)
        if os.path.exists(temp_path):
            os.remove(temp_path)
        return size

print("✅ XGBoost & LightGBM models defined")

## Cell 3: Deep Learning Models (PyTorch)

In [ ]:
# ─── PyTorch Helper Functions ────────────────────────────────────────────────
def train_pytorch_model(model, X_train, y_train, X_val, y_val, reshape_1d=False):
    # Impute NaNs with 0
    X_train_clean = X_train.fillna(0).values.astype(np.float32)
    X_val_clean = X_val.fillna(0).values.astype(np.float32)
    y_train_clean = y_train.values
    y_val_clean = y_val.values

    if reshape_1d:
        X_train_clean = X_train_clean.reshape(-1, 1, X_train_clean.shape[1])
        X_val_clean = X_val_clean.reshape(-1, 1, X_val_clean.shape[1])

    X_train_t = torch.tensor(X_train_clean, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_clean, dtype=torch.long)
    X_val_t = torch.tensor(X_val_clean, dtype=torch.float32)
    y_val_t = torch.tensor(y_val_clean, dtype=torch.long)

    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    best_val_loss = float('inf')
    patience = 10
    patience_counter = 0
    best_model_state = None

    start_time = time.time()
    epochs = 100
    
    for epoch in range(epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item() * batch_x.size(0)
        val_loss /= len(val_dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
                
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        
    train_time = time.time() - start_time
    return {"train_time": train_time, "epochs_trained": epoch + 1 - patience_counter}


def predict_pytorch_model(model, X, reshape_1d=False, proba=False):
    model.eval()
    X_clean = X.fillna(0).values.astype(np.float32)
    if reshape_1d:
        X_clean = X_clean.reshape(-1, 1, X_clean.shape[1])
        
    X_t = torch.tensor(X_clean, dtype=torch.float32)
    dataset = TensorDataset(X_t)
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    
    all_preds = []
    with torch.no_grad():
        for (batch_x,) in loader:
            batch_x = batch_x.to(DEVICE)
            outputs = model(batch_x)
            if proba:
                probs = torch.softmax(outputs, dim=1)
                all_preds.append(probs.cpu().numpy())
            else:
                preds = torch.argmax(outputs, dim=1)
                all_preds.append(preds.cpu().numpy())
                
    return np.concatenate(all_preds, axis=0)


# ─── 1D CNN Model ────────────────────────────────────────────────────────────
class CNN1DNetwork(nn.Module):
    def __init__(self, num_features, num_classes):
        super(CNN1DNetwork, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class TemporalCNN1DModel(BaseModel):
    def __init__(self):
        self.name = 'TemporalCNN1D'
        self.model = CNN1DNetwork(num_features=len(FEATURE_COLS), num_classes=NUM_CLASSES)

    def train(self, X_train, y_train, X_val, y_val) -> dict:
        return train_pytorch_model(self.model, X_train, y_train, X_val, y_val, reshape_1d=True)

    def predict(self, X) -> np.ndarray:
        return predict_pytorch_model(self.model, X, reshape_1d=True, proba=False)

    def predict_proba(self, X) -> np.ndarray:
        return predict_pytorch_model(self.model, X, reshape_1d=True, proba=True)

    def save(self, path: str):
        torch.save(self.model.state_dict(), path)

    def get_model_size_mb(self) -> float:
        temp_path = '/tmp/temp_cnn.pth'
        torch.save(self.model.state_dict(), temp_path)
        size = os.path.getsize(temp_path) / (1024 * 1024)
        if os.path.exists(temp_path):
            os.remove(temp_path)
        return size

print("✅ PyTorch 1D-CNN model defined")

## Cell 4: Feed-Forward MLP Model

In [ ]:
# ─── Feed-Forward MLP Model ──────────────────────────────────────────────────
class MLPNetwork(nn.Module):
    def __init__(self, num_features, num_classes):
        super(MLPNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)


class FeedForwardMLPModel(BaseModel):
    def __init__(self):
        self.name = 'FeedForwardMLP'
        self.model = MLPNetwork(num_features=len(FEATURE_COLS), num_classes=NUM_CLASSES)

    def train(self, X_train, y_train, X_val, y_val) -> dict:
        return train_pytorch_model(self.model, X_train, y_train, X_val, y_val, reshape_1d=False)

    def predict(self, X) -> np.ndarray:
        return predict_pytorch_model(self.model, X, reshape_1d=False, proba=False)

    def predict_proba(self, X) -> np.ndarray:
        return predict_pytorch_model(self.model, X, reshape_1d=False, proba=True)

    def save(self, path: str):
        torch.save(self.model.state_dict(), path)

    def get_model_size_mb(self) -> float:
        temp_path = '/tmp/temp_mlp.pth'
        torch.save(self.model.state_dict(), temp_path)
        size = os.path.getsize(temp_path) / (1024 * 1024)
        if os.path.exists(temp_path):
            os.remove(temp_path)
        return size

print("✅ PyTorch Feed-Forward MLP model defined")

## Cell 5: Tournament Utility Functions

In [ ]:
# ─── Data Splitting ──────────────────────────────────────────────────────────
def split_by_subject(df: pd.DataFrame, subject_col: str = 'subject_id'):
    """Splits data 70/15/15 by subject_id to prevent data leakage"""
    gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
    train_idx, temp_idx = next(gss1.split(df, groups=df[subject_col]))
    
    train_df = df.iloc[train_idx]
    temp_df = df.iloc[temp_idx]
    
    gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
    val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df[subject_col]))
    
    val_df = temp_df.iloc[val_idx]
    test_df = temp_df.iloc[test_idx]
    
    return train_df, val_df, test_df


# ─── Model Evaluation ────────────────────────────────────────────────────────
def evaluate_model(model: BaseModel, X_test: pd.DataFrame, y_test: pd.Series) -> dict:
    start_time = time.perf_counter()
    y_pred = model.predict(X_test)
    end_time = time.perf_counter()
    
    latency_ms = ((end_time - start_time) / len(X_test)) * 1000
    
    y_proba = model.predict_proba(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    mac_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    mac_prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    mac_rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    ll = log_loss(y_test, y_proba, labels=range(NUM_CLASSES))
    
    class_f1 = f1_score(y_test, y_pred, average=None, zero_division=0, labels=range(NUM_CLASSES))
    
    return {
        "Accuracy": acc,
        "Macro-F1": mac_f1,
        "Precision": mac_prec,
        "Recall": mac_rec,
        "Log-Loss": ll,
        "Latency(ms)": latency_ms,
        "Size(MB)": model.get_model_size_mb(),
        "Class_F1": {STATE_NAMES[i]: class_f1[i] for i in range(NUM_CLASSES)}
    }


# ─── Markdown Report Generation ──────────────────────────────────────────────
def generate_markdown_report(docs_dir: Path, stage_name: str, results: list, winner_name: str):
    docs_dir.mkdir(parents=True, exist_ok=True)
    report_path = docs_dir / 'MODEL_TOURNAMENT_RESULTS.md'
    
    is_new = not report_path.exists()
    
    with open(report_path, 'a') as f:
        if is_new:
            timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            f.write(f"# TRIDENT Multi-Model Tournament Results\n")
            f.write(f"> Auto-generated by `multi_model_tournament.py` on Kaggle\n")
            f.write(f"> Date: {timestamp}\n")
            f.write(f"> Device: {DEVICE} | GPU: {GPU_AVAILABLE}\n\n")
            
        f.write(f"## {stage_name}\n\n")
        f.write("| Model | Accuracy | Macro-F1 | Precision | Recall | Log-Loss | Latency(ms) | Size(MB) |\n")
        f.write("|---|---|---|---|---|---|---|---|\n")
        
        for res in results:
            f.write(f"| {res['model']} | {res['Accuracy']:.4f} | {res['Macro-F1']:.4f} | {res['Precision']:.4f} | "
                    f"{res['Recall']:.4f} | {res['Log-Loss']:.4f} | {res['Latency(ms)']:.4f} | {res['Size(MB)']:.2f} |\n")
            
        f.write("\n### Per-Class F1 Scores\n\n")
        header_classes = " | ".join(STATE_NAMES.values())
        header_sep = " | ".join(["---"] * len(STATE_NAMES))
        f.write(f"| Model | {header_classes} |\n")
        f.write(f"|---|{header_sep}|\n")
        
        for res in results:
            class_scores = " | ".join([f"{res['Class_F1'][c]:.4f}" for c in STATE_NAMES.values()])
            f.write(f"| {res['model']} | {class_scores} |\n")
            
        f.write(f"\n**Stage Winner(s) / Advancing:** {winner_name}\n\n")

print("✅ Utility functions defined")

## Cell 6: Load Data

In [ ]:
print(f"\n📂 Loading dataset from {DATA_PATH}...")
start_load = time.time()
df = pd.read_parquet(DATA_PATH)
load_time = time.time() - start_load

print(f"\n✅ Dataset loaded successfully")
print(f"   Shape: {df.shape}")
print(f"   Load time: {load_time:.2f}s")
print(f"   Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"   Subjects: {df['subject_id'].nunique()}")
print(f"   Classes: {df['health_state'].nunique()}")
print(f"\n   Class Distribution:")
for state_id, state_name in STATE_NAMES.items():
    count = (df['health_state'] == state_id).sum()
    pct = 100 * count / len(df)
    print(f"      {state_name}: {count:6d} ({pct:5.1f}%)")

## Cell 7: Stage 1 - Screening (10% Data)

In [ ]:
print(f"\n{'='*70}")
print(f"  STAGE 1: 10% SCREENING  ")
print(f"{'='*70}\n")

subjects = df['subject_id'].unique()
np.random.seed(42)
shuffled_subjects = np.random.permutation(subjects)

# Select 10% of subjects
n_subj_s1 = max(1, int(len(shuffled_subjects) * 0.10))
s1_subjects = shuffled_subjects[:n_subj_s1]
df_s1 = df[df['subject_id'].isin(s1_subjects)].copy()

print(f"Subjects in Stage 1: {len(s1_subjects)} / {len(subjects)} (10%)")
print(f"Samples in Stage 1: {len(df_s1)}\n")

# Split data
train_s1, val_s1, test_s1 = split_by_subject(df_s1)
X_tr, y_tr = train_s1[FEATURE_COLS], train_s1['health_state']
X_v, y_v = val_s1[FEATURE_COLS], val_s1['health_state']
X_te, y_te = test_s1[FEATURE_COLS], test_s1['health_state']

print(f"Train: {len(X_tr)} | Val: {len(X_v)} | Test: {len(X_te)}\n")

# Initialize all 4 models
models = [
    XGBoostModel(),
    LightGBMModel(),
    TemporalCNN1DModel(),
    FeedForwardMLPModel()
]

# Train all models
s1_results = []
for i, model in enumerate(models, 1):
    print(f"[{i}/4] Training {model.name}...", end=" ", flush=True)
    start = time.time()
    model.train(X_tr, y_tr, X_v, y_v)
    res = evaluate_model(model, X_te, y_te)
    res['model'] = model.name
    res['obj'] = model
    s1_results.append(res)
    elapsed = time.time() - start
    print(f"✅ ({elapsed:.1f}s) F1={res['Macro-F1']:.4f}")
    gc.collect()  # Free memory

# Rank by Macro F1
s1_results.sort(key=lambda x: x['Macro-F1'], reverse=True)
top_2 = s1_results[:2]
top_2_names = ", ".join([r['model'] for r in top_2])

print(f"\n🏆 Stage 1 Top 2: {top_2_names}")
print(f"   1. {s1_results[0]['model']}: {s1_results[0]['Macro-F1']:.4f}")
print(f"   2. {s1_results[1]['model']}: {s1_results[1]['Macro-F1']:.4f}")

generate_markdown_report(Path(DOCS_DIR), "Stage 1: 10% Screening", s1_results, top_2_names)

## Cell 8: Stage 2 - Semifinal (35% Data)

In [ ]:
print(f"\n{'='*70}")
print(f"  STAGE 2: 35% SEMIFINAL  ")
print(f"{'='*70}\n")

# Select 35% of subjects
n_subj_s2 = max(1, int(len(shuffled_subjects) * 0.35))
s2_subjects = shuffled_subjects[:n_subj_s2]
df_s2 = df[df['subject_id'].isin(s2_subjects)].copy()

print(f"Subjects in Stage 2: {len(s2_subjects)} / {len(subjects)} (35%)")
print(f"Samples in Stage 2: {len(df_s2)}\n")

# Split data
train_s2, val_s2, test_s2 = split_by_subject(df_s2)
X_tr, y_tr = train_s2[FEATURE_COLS], train_s2['health_state']
X_v, y_v = val_s2[FEATURE_COLS], val_s2['health_state']
X_te, y_te = test_s2[FEATURE_COLS], test_s2['health_state']

print(f"Train: {len(X_tr)} | Val: {len(X_v)} | Test: {len(X_te)}\n")

# Re-initialize top 2 models (fresh training)
s2_models = []
for r in top_2:
    if r['model'] == 'XGBoost':
        s2_models.append(XGBoostModel())
    elif r['model'] == 'LightGBM':
        s2_models.append(LightGBMModel())
    elif r['model'] == 'TemporalCNN1D':
        s2_models.append(TemporalCNN1DModel())
    elif r['model'] == 'FeedForwardMLP':
        s2_models.append(FeedForwardMLPModel())

# Train top 2 models
s2_results = []
for i, model in enumerate(s2_models, 1):
    print(f"[{i}/2] Training {model.name}...", end=" ", flush=True)
    start = time.time()
    model.train(X_tr, y_tr, X_v, y_v)
    res = evaluate_model(model, X_te, y_te)
    res['model'] = model.name
    res['obj'] = model
    s2_results.append(res)
    elapsed = time.time() - start
    print(f"✅ ({elapsed:.1f}s) F1={res['Macro-F1']:.4f}")
    gc.collect()

# Rank by Macro F1
s2_results.sort(key=lambda x: x['Macro-F1'], reverse=True)
top_1 = s2_results[0]
top_1_name = top_1['model']

print(f"\n🏆 Stage 2 Champion: {top_1_name}")
print(f"   Macro-F1: {top_1['Macro-F1']:.4f}")

generate_markdown_report(Path(DOCS_DIR), "Stage 2: 35% Semifinal", s2_results, top_1_name)

## Cell 9: Stage 3 - Final Championship (100% Data)

In [ ]:
print(f"\n{'='*70}")
print(f"  STAGE 3: 100% CHAMPIONSHIP  ")
print(f"{'='*70}\n")

print(f"Subjects: {len(subjects)} (100%)")
print(f"Samples: {len(df)}\n")

# Split full dataset
train_s3, val_s3, test_s3 = split_by_subject(df)
X_tr, y_tr = train_s3[FEATURE_COLS], train_s3['health_state']
X_v, y_v = val_s3[FEATURE_COLS], val_s3['health_state']
X_te, y_te = test_s3[FEATURE_COLS], test_s3['health_state']

print(f"Train: {len(X_tr)} | Val: {len(X_v)} | Test: {len(X_te)}\n")

# Re-initialize champion
if top_1['model'] == 'XGBoost':
    champion_model = XGBoostModel()
elif top_1['model'] == 'LightGBM':
    champion_model = LightGBMModel()
elif top_1['model'] == 'TemporalCNN1D':
    champion_model = TemporalCNN1DModel()
elif top_1['model'] == 'FeedForwardMLP':
    champion_model = FeedForwardMLPModel()

# Train champion
print(f"Training Champion {champion_model.name} on 100% data...", end=" ", flush=True)
start = time.time()
champion_model.train(X_tr, y_tr, X_v, y_v)
res = evaluate_model(champion_model, X_te, y_te)
res['model'] = champion_model.name
elapsed = time.time() - start
print(f"✅ ({elapsed:.1f}s)")

print(f"\n🏆 FINAL CHAMPION: {champion_model.name}")
print(f"   Accuracy:  {res['Accuracy']:.4f}")
print(f"   Macro-F1:  {res['Macro-F1']:.4f}")
print(f"   Precision: {res['Precision']:.4f}")
print(f"   Recall:    {res['Recall']:.4f}")
print(f"   Log-Loss:  {res['Log-Loss']:.4f}")
print(f"   Size:      {res['Size(MB)']:.2f} MB")
print(f"   Latency:   {res['Latency(ms)']:.4f} ms/sample")

generate_markdown_report(Path(DOCS_DIR), "Stage 3: 100% Championship", [res], champion_model.name)

gc.collect()

## Cell 10: Export Model & Results

In [ ]:
print(f"\n{'='*70}")
print(f"  EXPORTING ARTIFACTS  ")
print(f"{'='*70}\n")

# Create output directory
out_path = Path(OUTPUT_DIR)
out_path.mkdir(parents=True, exist_ok=True)

# Save model
final_model_path = out_path / 'wearable_health_model.pkl'
print(f"Saving model to {final_model_path}...")

if champion_model.name in ['TemporalCNN1D', 'FeedForwardMLP']:
    joblib.dump(champion_model, final_model_path)
else:
    champion_model.save(final_model_path)

model_size_mb = os.path.getsize(final_model_path) / (1024 * 1024)
print(f"✅ Model exported: {model_size_mb:.2f} MB")

# Results file
results_path = out_path / 'MODEL_TOURNAMENT_RESULTS.md'
print(f"\nResults markdown: {results_path}")
if results_path.exists():
    print(f"✅ File size: {os.path.getsize(results_path)} bytes")

print(f"\n🎉 Tournament Complete!")
print(f"\nFinal Output:")
print(f"   Model: {final_model_path.name}")
print(f"   Results: {results_path.name}")

## Cell 11: Verification & Summary

In [ ]:
print(f"\n{'='*70}")
print(f"  VERIFICATION & SUMMARY  ")
print(f"{'='*70}\n")

print(f"✅ Tournament Results Summary:\n")
print(f"Stage 1 (10% screening):")
for i, res in enumerate(sorted(s1_results, key=lambda x: x['Macro-F1'], reverse=True), 1):
    print(f"   {i}. {res['model']:16s} F1={res['Macro-F1']:.4f} Acc={res['Accuracy']:.4f}")

print(f"\nStage 2 (35% semifinal):")
for i, res in enumerate(sorted(s2_results, key=lambda x: x['Macro-F1'], reverse=True), 1):
    print(f"   {i}. {res['model']:16s} F1={res['Macro-F1']:.4f} Acc={res['Accuracy']:.4f}")

print(f"\nStage 3 (100% championship):")
print(f"   🏆 {champion_model.name:16s} F1={res['Macro-F1']:.4f} Acc={res['Accuracy']:.4f}")

print(f"\n📊 Deliverables:")
print(f"   ✅ Model artifact: wearable_health_model.pkl ({model_size_mb:.2f} MB)")
print(f"   ✅ Results report: MODEL_TOURNAMENT_RESULTS.md")
print(f"   ✅ Execution time: {time.time() - start_load:.1f}s (total)")
print(f"\n🎯 Next Steps:")
print(f"   1. Download 'wearable_health_model.pkl' from /kaggle/working/")
print(f"   2. Download 'MODEL_TOURNAMENT_RESULTS.md' from /kaggle/working/")
print(f"   3. Copy to GitHub repo: src/ml/wearable_health_model.pkl")
print(f"   4. Copy to GitHub repo: docs/MODEL_TOURNAMENT_RESULTS.md")